# Coleta de imagens de fósseis do MNHN pelo GBIF

Baixa as imagens da [coleção de fósseis do Muséum national d'Histoire naturelle](https://www.gbif.org/dataset/b275a4c1-9859-4f3c-8ead-d86dde820fbc) (Paris) publicada no GBIF e salva no Google Drive, no formato que `scripts/preparar_splits.py` espera:

```
<PASTA_DRIVE>/
  imagens/<filo>/<gbifID>.jpg     uma imagem por espécime
  metadados_gbif.csv              taxonomia, era, localidade, URL, licença e autor de cada registro
  downloads.csv                   status do download de cada imagem
```

Observações sobre os dados (consulta de novembro de 2026):

- 138.792 registros têm imagem, mas só cerca de 58 mil têm filo na taxonomia do GBIF. O MNHN envia apenas o nome científico, e muitos espécimes não têm identificação. O rótulo usado aqui é o filo interpretado pelo GBIF.
- Muitos espécimes têm várias fotos. Por padrão salvamos só a primeira (`IMAGENS_POR_ESPECIME = 1`), para que cada arquivo continue correspondendo a um espécime e o split por arquivo não coloque fotos do mesmo fóssil em treino e teste.
- As imagens estão sob licença CC BY 4.0. O CSV guarda o autor (`creator`) e a licença de cada imagem, necessários para a atribuição.

O notebook pode ser executado de novo sem perder progresso: a consulta fica salva em `metadados_gbif.csv` e as imagens já baixadas são puladas.

## 1. Configuração

In [ ]:
DATASET_KEY = "b275a4c1-9859-4f3c-8ead-d86dde820fbc"
PASTA_DRIVE = "/content/drive/MyDrive/Image_Classifier_Fossil/gbif_mnhn"

# Filos a coletar (mesmo problema de 2021). Outros exemplos: "Echinodermata", "Cnidaria", "Brachiopoda".
FILOS = ["Arthropoda", "Bryozoa", "Mollusca"]

# None baixa tudo. Use um número pequeno (ex: 30) na primeira execução para testar.
LIMITE_POR_CLASSE = 30

IMAGENS_POR_ESPECIME = 1   # manter 1 para o split por arquivo ser um split por espécime
LADO_MAX = 512             # maior lado da imagem salva, em pixels (None mantém o original)
N_THREADS = 8              # downloads em paralelo; valores altos podem sobrecarregar o servidor do museu
REFAZER_CONSULTA = False   # True ignora o metadados_gbif.csv salvo e consulta a API de novo
SEED = 42

# Identificação enviada nas requisições. Se quiser, inclua um contato.
USER_AGENT = "fossil-image-research/1.0 (coleta academica de imagens do GBIF)"

## 2. Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import io
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm

API = "https://api.gbif.org/v1"
PASTA = Path(PASTA_DRIVE)
PASTA_IMAGENS = PASTA / "imagens"
CSV_METADADOS = PASTA / "metadados_gbif.csv"
CSV_DOWNLOADS = PASTA / "downloads.csv"
PASTA_IMAGENS.mkdir(parents=True, exist_ok=True)

sessao = requests.Session()
sessao.headers["User-Agent"] = USER_AGENT

## 3. Consulta à API do GBIF

A busca de ocorrências do GBIF devolve no máximo 300 registros por página e não pagina além de 100 mil. Como nenhum filo passa disso, a consulta é feita filo a filo.

In [ ]:
CAMPOS = [
    "kingdom", "phylum", "class", "order", "family", "genus", "species",
    "scientificName", "taxonRank", "typeStatus", "catalogNumber", "occurrenceID",
    "country", "stateProvince", "municipality", "locality",
    "earliestEraOrLowestErathem", "earliestPeriodOrLowestSystem",
    "earliestEpochOrLowestSeries", "earliestAgeOrLowestStage",
    "preparations", "recordedBy", "identifiedBy",
]


def get_json(url, params=None, tentativas=4):
    for i in range(tentativas):
        try:
            r = sessao.get(url, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if i == tentativas - 1:
                raise
            time.sleep(2 ** i)


def chave_do_filo(nome):
    d = get_json(f"{API}/species/match", {"name": nome, "rank": "PHYLUM", "strict": "true"})
    if d.get("rank") != "PHYLUM" or "usageKey" not in d:
        raise ValueError(f"Filo não encontrado no GBIF: {nome}")
    return d["usageKey"]


def imagens_do_registro(registro):
    return [
        m for m in registro.get("media", [])
        if m.get("type") == "StillImage" and m.get("identifier")
    ]


def consultar_filo(nome, limite_paginas=None):
    chave = chave_do_filo(nome)
    params = {"datasetKey": DATASET_KEY, "mediaType": "StillImage", "phylumKey": chave, "limit": 300}
    linhas, offset, total = [], 0, None
    with tqdm(desc=nome, unit=" registros") as barra:
        while True:
            d = get_json(f"{API}/occurrence/search", {**params, "offset": offset})
            if total is None:
                total = d["count"]
                barra.total = total
            for r in d["results"]:
                imgs = imagens_do_registro(r)
                for ordem, m in enumerate(imgs[:IMAGENS_POR_ESPECIME]):
                    linhas.append({
                        "gbifID": r["key"],
                        "ordem_imagem": ordem,
                        "n_imagens_especime": len(imgs),
                        **{c: r.get(c) for c in CAMPOS},
                        "url_imagem": m["identifier"],
                        "licenca_imagem": m.get("license") or r.get("license"),
                        "autor_imagem": m.get("creator"),
                        "detentor_direitos": m.get("rightsHolder"),
                    })
            barra.update(len(d["results"]))
            offset += len(d["results"])
            if d["endOfRecords"] or not d["results"]:
                break
            if limite_paginas and offset >= limite_paginas * 300:
                break
    return pd.DataFrame(linhas)


if CSV_METADADOS.exists() and not REFAZER_CONSULTA:
    metadados = pd.read_csv(CSV_METADADOS)
    print(f"Metadados carregados de {CSV_METADADOS}")
else:
    metadados = pd.concat([consultar_filo(f) for f in FILOS], ignore_index=True)
    metadados.to_csv(CSV_METADADOS, index=False)
    print(f"Metadados salvos em {CSV_METADADOS}")

print(f"{metadados['gbifID'].nunique()} espécimes, {len(metadados)} imagens")
metadados.groupby("phylum")["gbifID"].nunique().rename("especimes").to_frame()

Classes dentro de cada filo (útil para o problema de classe em Mollusca):

In [ ]:
(
    metadados.drop_duplicates("gbifID")
    .fillna({"class": "Sem classe"})
    .groupby(["phylum", "class"]).size()
    .rename("especimes").to_frame()
    .sort_values(["phylum", "especimes"], ascending=[True, False])
)

## 4. Seleção do que baixar

Com `LIMITE_POR_CLASSE`, sorteia esse número de espécimes por filo. O caminho de cada imagem é `imagens/<filo>/<gbifID>.jpg` (com sufixo `_<n>` se `IMAGENS_POR_ESPECIME > 1`).

In [ ]:
def caminho_imagem(linha):
    sufixo = f"_{linha.ordem_imagem}" if linha.ordem_imagem else ""
    return PASTA_IMAGENS / str(linha.phylum) / f"{linha.gbifID}{sufixo}.jpg"


selecao = metadados.dropna(subset=["phylum", "url_imagem"])
if LIMITE_POR_CLASSE:
    especimes = (
        selecao.drop_duplicates("gbifID")
        .sample(frac=1, random_state=SEED)
        .groupby("phylum").head(LIMITE_POR_CLASSE)["gbifID"]
    )
    selecao = selecao[selecao["gbifID"].isin(especimes)]
selecao = selecao.assign(caminho=[str(caminho_imagem(l)) for l in selecao.itertuples()])
pendentes = selecao[~selecao["caminho"].map(lambda c: Path(c).exists())]
print(f"Selecionadas: {len(selecao)} imagens. Já no Drive: {len(selecao) - len(pendentes)}. Pendentes: {len(pendentes)}")

## 5. Download

Cada imagem é baixada, convertida para RGB, reduzida para `LADO_MAX` e salva em JPEG. Falhas não interrompem a coleta: ficam registradas em `downloads.csv` e são tentadas de novo na próxima execução.

Antes do download completo, a célula abaixo testa o acesso com 5 imagens. Se todas falharem com 403, o servidor de imagens do museu está recusando as requisições deste ambiente; nesse caso, não insista com muitas tentativas.

In [ ]:
def baixar(url, destino, tentativas=3):
    erro = None
    for i in range(tentativas):
        try:
            r = sessao.get(url, timeout=60)
            if r.status_code in (403, 404, 410):
                return f"http_{r.status_code}"
            r.raise_for_status()
            img = Image.open(io.BytesIO(r.content)).convert("RGB")
            if LADO_MAX:
                img.thumbnail((LADO_MAX, LADO_MAX))
            destino = Path(destino)
            destino.parent.mkdir(parents=True, exist_ok=True)
            temporario = destino.with_suffix(".tmp")
            img.save(temporario, "JPEG", quality=90)
            temporario.replace(destino)  # evita arquivo pela metade se a execução cair
            return "ok"
        except Exception as e:  # noqa: BLE001
            erro = type(e).__name__
            time.sleep(2 ** i)
    return f"erro_{erro}"


teste = pendentes.head(5) if len(pendentes) else selecao.head(5)
resultado_teste = []
for linha in teste.itertuples():
    s = "ok" if Path(linha.caminho).exists() else baixar(linha.url_imagem, linha.caminho)
    resultado_teste.append(s)
    print(linha.gbifID, linha.phylum, s)

if resultado_teste and all(s == "http_403" for s in resultado_teste):
    raise RuntimeError(
        "O servidor de imagens do MNHN recusou todas as requisições (403, proteção da Cloudflare). "
        "Não rode o download completo: veja a seção 'Se o download der 403' no fim do notebook."
    )

In [ ]:
def registrar(resultados):
    novo = pd.DataFrame(resultados, columns=["gbifID", "ordem_imagem", "caminho", "status"])
    if CSV_DOWNLOADS.exists():
        antigo = pd.read_csv(CSV_DOWNLOADS)
        novo = pd.concat([antigo, novo]).drop_duplicates(["gbifID", "ordem_imagem"], keep="last")
    novo.to_csv(CSV_DOWNLOADS, index=False)


pendentes = selecao[~selecao["caminho"].map(lambda c: Path(c).exists())]
resultados = []
with ThreadPoolExecutor(N_THREADS) as pool:
    futuros = {
        pool.submit(baixar, l.url_imagem, l.caminho): (l.gbifID, l.ordem_imagem, l.caminho)
        for l in pendentes.itertuples()
    }
    for i, f in enumerate(tqdm(as_completed(futuros), total=len(futuros), desc="Imagens"), 1):
        resultados.append((*futuros[f], f.result()))
        if i % 500 == 0:  # salva o progresso de tempos em tempos
            registrar(resultados)
            resultados = []
registrar(resultados)

status = pd.read_csv(CSV_DOWNLOADS)
print(status["status"].value_counts().to_string())

Imagens no Drive por filo:

In [ ]:
pd.Series({p.name: len(list(p.glob("*.jpg"))) for p in sorted(PASTA_IMAGENS.iterdir()) if p.is_dir()}, name="imagens")

## 6. Amostra: 3 imagens por filo

In [ ]:
N_AMOSTRA = 3

baixadas = selecao[selecao["caminho"].map(lambda c: Path(c).exists())]
if baixadas.empty:
    raise RuntimeError("Nenhuma imagem baixada ainda. Rode a seção 5 antes.")
amostra = baixadas.sample(frac=1, random_state=SEED).groupby("phylum").head(N_AMOSTRA)
filos = sorted(amostra["phylum"].unique())

fig, eixos = plt.subplots(len(filos), N_AMOSTRA, figsize=(4 * N_AMOSTRA, 4.3 * len(filos)), squeeze=False)
for i, filo in enumerate(filos):
    linhas = amostra[amostra["phylum"] == filo]
    for j, ax in enumerate(eixos[i]):
        ax.axis("off")
        if j >= len(linhas):
            continue
        l = linhas.iloc[j]
        ax.imshow(Image.open(l["caminho"]))
        nome = l["scientificName"] if isinstance(l["scientificName"], str) else "sem identificação"
        classe = l["class"] if isinstance(l["class"], str) else "sem classe"
        ax.set_title(f"{filo} | {classe}\n{nome[:40]}\ngbifID {l['gbifID']}", fontsize=9)
fig.suptitle("MNHN, coleção de fósseis (F), via GBIF. Imagens CC BY 4.0, autores em metadados_gbif.csv", fontsize=10)
plt.tight_layout()
plt.show()

## 7. Próximo passo

A pasta `imagens/` já está no formato do `preparar_splits.py` (uma subpasta por classe). Com o repositório clonado no Colab:

```
!python scripts/preparar_splits.py --imagens "/content/drive/MyDrive/Image_Classifier_Fossil/gbif_mnhn/imagens" --saida data/splits_gbif_filo.csv
!python scripts/treinar_baseline.py --splits data/splits_gbif_filo.csv --saida modelos/gbif_filo_cnn
```

Para coletar tudo, mude `LIMITE_POR_CLASSE = None` e rode de novo a partir da seção 4. Os metadados textuais em `metadados_gbif.csv` (taxonomia, era, localidade) ficam disponíveis para os experimentos com VLMs.

## Se o download der 403

As imagens ficam em `mediaphoto.mnhn.fr`, que está atrás da proteção anti-bot da Cloudflare. Em novembro de 2026 esse servidor respondeu 403 a requisições automáticas, embora a consulta de metadados pelo GBIF funcionasse. Nesse caso, não tente contornar o bloqueio. O caminho é pedir acesso:

- ao MNHN, pedindo acesso em lote às imagens da coleção F para pesquisa acadêmica (as imagens já são CC BY 4.0);
- ao helpdesk do GBIF (helpdesk@gbif.org), perguntando se há uma forma oficial de baixar em lote as mídias deste dataset.

Os metadados (`metadados_gbif.csv`) continuam válidos e podem ser usados assim que as imagens estiverem disponíveis.